# Phase B: Kaggle Retraining with Fixed Detector Gradient Mappings

This notebook retrains both the WheelEye-Classify and WheelEye-Detect models on Kaggle. It applies the fixes for the gradient misalignment that was preventing the loss from converging.

In [ ]:
!pip install torch torchvision numpy Pillow onnx onnxruntime tqdm

## 1. Clone the repository
Bring the latest fixed code into the Kaggle environment.

In [ ]:
!git clone https://github.com/GughanS/renault_nissan.git
%cd renault_nissan

## 2. Generate / Prepare Dataset
Generate the synthetic dataset mimicking the factory conditions, as well as the classifier crops dataset.

In [ ]:
# Generate 1000 synthetic images and labels in YOLO format for detector AND crops for classifier
!python scripts/generate_synthetic_data.py --num-images 1000 --out-dir ./data --task both

## 3. Train the Classifier
Train the WheelEye Classifier on the crops.

In [ ]:
!python scripts/train_classify.py --img-dir ./data/crops --csv-path ./data/crops_labels.csv --epochs 20 --batch-size 32

## 4. Train the Detector
Train the fixed WheelEye Detector on the full images. Note the fixed `permute` and data loading distribution.

In [ ]:
!python scripts/train_detect.py --img-dir ./data/images --label-dir ./data/labels --epochs 50 --batch-size 16

## 5. Export to ONNX
Export both models to ONNX to prepare them for deployment in the Docker container.

In [ ]:
!python scripts/export_onnx.py

## 6. Run ONNX Parity Tests
Ensure the exported ONNX models exactly match the PyTorch models numerically.

In [ ]:
!python tests/test_onnx_parity.py